# Uttarakhand (S28) roll ingestion — hub-free runner
Drives `uk/uk_driver.py`. **0 MB on the Mac** — all work is here; records stream to Google Drive as private Parquet, PDFs are discarded.

Order: **check** (self-test, no Drive needed) → **parts** (exact download size) → **textlayer** (2027 records). 2026 OCR is a separate (CC#2) stage.

In [ ]:
# partial clone (blob:none = lazy fetch; the 500MB OCR models are never pulled for textlayer) + deps
!git clone --depth 1 --filter=blob:none https://github.com/aditya55496/eci-rolls.git /content/eci-rolls
!pip -q install requests cryptography pymupdf pyarrow

In [ ]:
%cd /content/eci-rolls
# self-test (no Drive mount needed): part-list -> 1 PDF -> parse -> record count (no names printed)
!python uk/uk_driver.py check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/voter-engine/data'

In [ ]:
# exact download size: part-list sweep across all 70 ACs x 3 rolls
!python uk/uk_driver.py parts --out "$OUT"
import json; print(json.load(open(OUT + '/manifest.json')))

In [ ]:
# 2027 SSR records (text layer, ~100%, no OCR) -> private Parquet per AC
!python uk/uk_driver.py textlayer --out "$OUT" --acs 1-70